**Title:** Beyond the Box: Milestone Four Improvement/Iteration Run  
**Author:** William Anderson  
**Date:** 8 August 2026  
**Description:** Trade-focused fine-tuning improvement + Sample RAG Demo

#### Sample RAG Demo

This is intentionally a **Sample RAG Demo**, not me pretending I built some massive production retrieval system in the last week of class - but I figured I would do it regardless, given that I've talked about including it and you also gave me feedback about it. The SQLite database already has player dimensions and player level attributes, so I'm using a small structured retrieval step to validate a player name and grab a recent player snapshot before Qwen writes the response. Also, this is using the fine tuned model of the new Qwen 1.5B that was done in Google Colab. 

This is also a pretty natural extension of what the earlier milestones showed: fine-tuning can help with behavior and formatting, but it can't responsibly create facts that weren't supplied; retrieval, instead, can provide some of those facts first.

In [1]:
from pathlib import Path
import sqlite3
from difflib import get_close_matches

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

PROJECT_DIR = Path(r"C:\BU MS DS\DSC670 - Adv Uses GAI")
NBA_DB_PATH = PROJECT_DIR / "nba_stats" / "nba.sqlite"
MODEL_PATH = PROJECT_DIR / "qwen-nba-1.5b-full-sft-v1"

assert NBA_DB_PATH.exists(), (
    f"NBA SQLite database not found: {NBA_DB_PATH}"
)

assert MODEL_PATH.exists(), (
    f"Fine-tuned model not found: {MODEL_PATH}"
)

DEVICE = (
    torch.device("xpu")
    if torch.xpu.is_available()
    else torch.device("cpu")
)

if DEVICE.type == "xpu":
    DTYPE = (
        torch.bfloat16
        if torch.xpu.is_bf16_supported()
        else torch.float16
    )
else:
    DTYPE = torch.float32

print("NBA SQLite:", NBA_DB_PATH)
print("Fine-tuned model:", MODEL_PATH)
print("Inference device:", DEVICE)

NBA SQLite: C:\BU MS DS\DSC670 - Adv Uses GAI\nba_stats\nba.sqlite
Fine-tuned model: C:\BU MS DS\DSC670 - Adv Uses GAI\qwen-nba-1.5b-full-sft-v1
Inference device: xpu


In [2]:
SYSTEM_PROMPT = (
    "You are an NBA narrative writing assistant for a public-facing sports analysis app. "
    "Use only the information supplied by the user or explicitly retrieved by the application. "
    "Don't use outside knowledge, even if you believe you know it. Don't invent players, teams, "
    "statistics, injuries, rumors, betting lines, salary-cap details, contract details, roster "
    "history, draft picks, or transactions. Preserve player names, team names, scores, and trade "
    "assets exactly as supplied. If a trade question is vague and the teams or package are missing, "
    "say that the information is missing instead of creating a destination or package. When a "
    "complete trade package is supplied, discuss every team involved and give tentative grades. "
    "When a potential trade is proposed, clearly label it as hypothetical and never present it as "
    "completed news. For game statistics, don't claim that a statistic favored the winner when the "
    "supplied numbers show the opposite. Write clearly and conversationally, using natural "
    "contractions where appropriate. Basically, don't hallucinate and don't rewrite the facts."
)

print(SYSTEM_PROMPT)

You are an NBA narrative writing assistant for a public-facing sports analysis app. Use only the information supplied by the user or explicitly retrieved by the application. Don't use outside knowledge, even if you believe you know it. Don't invent players, teams, statistics, injuries, rumors, betting lines, salary-cap details, contract details, roster history, draft picks, or transactions. Preserve player names, team names, scores, and trade assets exactly as supplied. If a trade question is vague and the teams or package are missing, say that the information is missing instead of creating a destination or package. When a complete trade package is supplied, discuss every team involved and give tentative grades. When a potential trade is proposed, clearly label it as hypothetical and never present it as completed news. For game statistics, don't claim that a statistic favored the winner when the supplied numbers show the opposite. Write clearly and conversationally, using natural contr

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH)
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

improved_model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_PATH),
    dtype=DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE)

improved_model.eval()
improved_model.config.use_cache = True

print("Model loaded for inference.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded for inference.


In [5]:
def generate_response(
    current_model,
    user_message,
    max_new_tokens=260,
):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_message,
        },
    ]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    model_inputs = model_inputs.to(
        next(
            current_model.parameters()
        ).device
    )

    prompt_length = (
        model_inputs["input_ids"]
        .shape[-1]
    )

    with torch.inference_mode():
        output_ids = current_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    response_ids = output_ids[
        0,
        prompt_length:,
    ]

    return tokenizer.decode(
        response_ids,
        skip_special_tokens=True,
    ).strip()

In [6]:
def get_sqlite_connection():
    connection = sqlite3.connect(
        str(NBA_DB_PATH)
    )

    connection.row_factory = sqlite3.Row

    connection.execute(
        "PRAGMA query_only = ON"
    )

    return connection


def get_table_columns(
    connection,
    table_name,
):
    rows = connection.execute(
        f'PRAGMA table_info("{table_name}")'
    ).fetchall()

    return [
        row["name"]
        for row in rows
    ]


def find_player(
    connection,
    player_name,
):
    player_name = player_name.strip()

    exact_row = connection.execute(
        """
        SELECT
            id,
            full_name,
            first_name,
            last_name,
            is_active
        FROM player
        WHERE LOWER(full_name) = LOWER(?)
        LIMIT 1
        """,
        (player_name,),
    ).fetchone()

    if exact_row:
        return {
            "player_id": exact_row["id"],
            "player_name": exact_row["full_name"],
            "first_name": exact_row["first_name"],
            "last_name": exact_row["last_name"],
            "is_active": exact_row["is_active"],
            "match_type": "exact",
        }

    player_rows = connection.execute(
        """
        SELECT
            id,
            full_name,
            first_name,
            last_name,
            is_active
        FROM player
        WHERE full_name IS NOT NULL
        """
    ).fetchall()

    player_names = [
        row["full_name"]
        for row in player_rows
        if row["full_name"]
    ]

    close_matches = get_close_matches(
        player_name,
        player_names,
        n=1,
        cutoff=0.70,
    )

    if not close_matches:
        return None

    matched_name = close_matches[0]

    matched_row = next(
        row
        for row in player_rows
        if row["full_name"] == matched_name
    )

    return {
        "player_id": matched_row["id"],
        "player_name": matched_row["full_name"],
        "first_name": matched_row["first_name"],
        "last_name": matched_row["last_name"],
        "is_active": matched_row["is_active"],
        "match_type": "fuzzy",
    }


def retrieve_common_player_info(
    connection,
    player_id,
):
    available_columns = get_table_columns(
        connection,
        "common_player_info",
    )

    wanted_columns = [
        "display_first_last",
        "birthdate",
        "school",
        "country",
        "height",
        "weight",
        "season_exp",
        "jersey",
        "position",
        "rosterstatus",
        "team_id",
        "team_name",
        "team_abbreviation",
        "team_code",
        "team_city",
        "from_year",
        "to_year",
        "draft_year",
        "draft_round",
        "draft_number",
        "greatest_75_flag",
    ]

    selected_columns = [
        column
        for column in wanted_columns
        if column in available_columns
    ]

    if not selected_columns:
        return {}

    select_clause = ", ".join(
        f'"{column}"'
        for column in selected_columns
    )

    row = connection.execute(
        f"""
        SELECT
            {select_clause}
        FROM common_player_info
        WHERE CAST(person_id AS TEXT) = CAST(? AS TEXT)
        LIMIT 1
        """,
        (player_id,),
    ).fetchone()

    if row is None:
        return {}

    return {
        column: row[column]
        for column in selected_columns
        if row[column] is not None
    }


def retrieve_player_context(
    player_name,
):
    with get_sqlite_connection() as connection:
        player_match = find_player(
            connection,
            player_name,
        )

        if player_match is None:
            return None

        profile = retrieve_common_player_info(
            connection,
            player_match["player_id"],
        )

        return {
            **player_match,
            "source_table": "common_player_info",
            "profile": profile,
        }


def context_to_text(
    context,
):
    if context is None:
        return (
            "No player match was found "
            "in the local NBA database."
        )

    lines = [
        f"Validated player: {context['player_name']}",
        f"Player ID: {context['player_id']}",
        f"Match type: {context['match_type']}",
        (
            "Active player flag: "
            f"{context['is_active']}"
        ),
    ]

    readable_labels = {
        "display_first_last": "Player name",
        "birthdate": "Birthdate",
        "school": "School",
        "country": "Country",
        "height": "Height",
        "weight": "Weight",
        "season_exp": "NBA experience",
        "jersey": "Jersey",
        "position": "Position",
        "rosterstatus": "Roster status",
        "team_id": "Team ID",
        "team_name": "Team",
        "team_abbreviation": "Team abbreviation",
        "team_code": "Team code",
        "team_city": "Team city",
        "from_year": "NBA from year",
        "to_year": "NBA through year",
        "draft_year": "Draft year",
        "draft_round": "Draft round",
        "draft_number": "Draft number",
        "greatest_75_flag": "NBA 75 flag",
    }

    profile = context.get(
        "profile",
        {},
    )

    if profile:
        lines.append(
            "Retrieved player context:"
        )

        for key, value in profile.items():
            label = readable_labels.get(
                key,
                key,
            )

            lines.append(
                f"{label}: {value}"
            )

    return "\n".join(lines)

##### Sample Player Retrieval

Anthony Davis is a useful test here because that potential-trade holdout was one of the worst Milestone 3 responses (though arguably the best example at the same time). This retrieval isn't supposed to tell Qwen whether a trade happened - it just validates the player and supplies some player context from the local/import DB source from kaggle.

In [7]:
assert NBA_DB_PATH.exists(), (
    f"NBA SQLite database not found: {NBA_DB_PATH}"
)

sample_player_context = (
    retrieve_player_context(
        "Anthony Davis"
    )
)

print(
    context_to_text(
        sample_player_context
    )
)

Validated player: Anthony Davis
Player ID: 203076
Match type: exact
Active player flag: 1
Retrieved player context:
Player name: Anthony Davis
Birthdate: 1993-03-11 00:00:00
School: Kentucky
Country: USA
Height: 6-10
Weight: 253
NBA experience: 11.0
Jersey: 3
Position: Forward-Center
Roster status: Active
Team ID: 1610612747
Team: Lakers
Team abbreviation: LAL
Team code: lakers
Team city: Los Angeles
NBA from year: 2012.0
NBA through year: 2023.0
Draft year: 2012
Draft round: 1
Draft number: 1
NBA 75 flag: Y


##### Sample RAG Generation

The retrieved context is added separately from the user's hypothetical package so Qwen can use the player information without changing the transaction itself - of course, the trade still needs to remain hypothetical.

In [8]:
rag_demo_prompt = f"""
USER REQUEST:
Potential trade idea: Oklahoma City receives Anthony Davis from the Lakers for one young starter and three first-round picks. Treat this only as a hypothetical and grade both teams.

SAMPLE RAG CONTEXT FROM THE LOCAL NBA DATABASE:
{context_to_text(sample_player_context)}

RULES:
Use the retrieved player context only as supporting context.
Preserve Anthony Davis, Oklahoma City, the Lakers, one young starter, and three first-round picks exactly.
Do not add another player or pick.
Do not claim that this trade happened.
"""

rag_demo_response = generate_response(
    improved_model,
    rag_demo_prompt,
)

print("Sample RAG Demo response:\n")
print(rag_demo_response)

Sample RAG Demo response:

Oklahoma City: B. The Thunder receive Anthony Davis in exchange for one young starter and three first-round picks. Lakers: B+. Los Angeles gets the young starter and picks. This is a hypothetical only, based only on the supplied information.


honestly, pretty good response for qwen 1.5b after fine tuning it in google colab due to GPU data restrictions on my local computer